<a href="https://colab.research.google.com/github/nithin12342/data-analysis-and-2026-ai-projection./blob/master/DATA_ACQUISITION_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Acquisition Pipeline - Google Colab

This notebook automates cloning the repository and running the 54-table data acquisition pipeline using Google Colab. The database will be stored persistently in your Google Drive.

## Step 1: Clone Repository

Run the cell below to clone the repository to your Colab session. If the repository is public, you can leave the token field blank.

In [1]:
#@title GitHub Authentication (Optional)
#@markdown Enter your GitHub Personal Access Token (PAT) if the repository is private (otherwise leave it blank):
github_token = "" #@param {type:"string"}

if github_token:
    !git clone https://{github_token}@github.com/nithin12342/data-analysis-and-2026-ai-projection..git data-analysis-and-2026-ai-projection
else:
    !git clone https://github.com/nithin12342/data-analysis-and-2026-ai-projection..git data-analysis-and-2026-ai-projection

%cd data-analysis-and-2026-ai-projection

Cloning into 'data-analysis-and-2026-ai-projection'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'data-analysis-and-2026-ai-projection'
/content


## Step 2: Upload Relevant Project Files (Optional)

Because folders like `DATA/` (containing SEC DERA source data) and files like `.env` (containing API keys) are typically git-ignored, you can upload them to your Colab session here so the pipeline can use them.

### Download SEC DERA Quarterly Datasets

If you do not have the SEC DERA dataset already saved on Google Drive, run this cell to download all 13 quarters (2023Q1 to 2026Q1) directly from the SEC website into the `DATA/` directory.

In [ ]:
#@title Download & Extract SEC DERA Data
import os
import zipfile
import urllib.request
import time

data_dir = "/content/data-analysis-and-2026-ai-projection/DATA"
os.makedirs(data_dir, exist_ok=True)

quarters = [
    "2023q1", "2023q2", "2023q3", "2023q4",
    "2024q1", "2024q2", "2024q3", "2024q4",
    "2025q1", "2025q2", "2025q3", "2025q4",
    "2026q1"
]

headers = {"User-Agent": "AntigravityResearch/1.0 (contact@antigravity.ai)"}

print("Starting download of 13 quarters of SEC DERA data (approx 200MB total)...\n")

for q in quarters:
    q_dir = os.path.join(data_dir, q)
    # Skip if sub.txt already exists to avoid redundant downloads
    if os.path.exists(os.path.join(q_dir, "sub.txt")):
        print(f"[{q}] Already exists. Skipping.")
        continue
        
    os.makedirs(q_dir, exist_ok=True)
    zip_url = f"https://www.sec.gov/files/dera/data/financial-statement-data-sets/{q}.zip"
    zip_path = os.path.join(data_dir, f"{q}.zip")
    
    print(f"[{q}] Downloading from SEC...")
    try:
        req = urllib.request.Request(zip_url, headers=headers)
        with urllib.request.urlopen(req) as response:
            with open(zip_path, 'wb') as out_file:
                out_file.write(response.read())
                
        print(f"[{q}] Extracting zip...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(q_dir)
            
        os.remove(zip_path)
        print(f"[{q}] Ingested successfully.\n")
        time.sleep(0.15)  # Respect SEC rate limits (10 reqs/sec max)
    except Exception as e:
        print(f"[{q}] Failed: {e}\n")

print("SEC DERA download process complete.")

In [ ]:
#@title Mount Google Drive & Copy `DATA` Folder
#@markdown If you have your `DATA` folder (containing SEC quarterly folders) saved on Google Drive, run this cell to mount Drive and copy it to the workspace:
from google.colab import drive
import os

drive.mount('/content/drive')

drive_data_path = '/content/drive/MyDrive/DATA'
if os.path.exists(drive_data_path):
    print("Copying DATA folder from Google Drive...")
    !cp -r /content/drive/MyDrive/DATA /content/data-analysis-and-2026-ai-projection/
    print("DATA folder copied successfully!")
else:
    print("DATA folder not found in Google Drive. You can manually upload SEC files via the Colab files tab or skip if fetching data from web.")

## Step 3: Install Requirements

Install the required libraries (DuckDB, Pandas, requests, etc.) to set up the ingestion environment.

In [ ]:
# Install dependencies
!pip install -r scripts/acquisition/requirements.txt

## Step 4: Run Ingestion Pipeline

Run the orchestrator script. Adding `--mount-drive` will prompt you to connect Google Drive, enabling the script to persistently save the DuckDB database to your Drive root under `master_consolidated.duckdb` so you can retrieve it later.

In [ ]:
# Run all data acquisition clusters (A, B, D, E) and build derived tables
!python scripts/acquisition/colab_pipeline.py --cluster all --mount-drive

## Step 5: Verify Database Ingestion

After ingestion completes, you can run this cell to verify that the tables were created successfully and count the number of rows populated in each.

In [ ]:
import duckdb
import os

# Connect to the database (check Google Drive first, then fallback to local path)
db_path = "/content/drive/MyDrive/master_consolidated.duckdb"
if not os.path.exists(db_path):
    local_path = "/content/data-analysis-and-2026-ai-projection/databases/master_consolidated.duckdb"
    if os.path.exists(local_path):
        db_path = local_path
        print(f"Using local database: {db_path}")
    else:
        print(f"No database found at {db_path} or {local_path}")

try:
    con = duckdb.connect(db_path)
    tables = con.execute("SHOW TABLES").fetchall()
    print(f"Database verified successfully. Found {len(tables)} tables:")
    for t in tables:
        count = con.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0]
        print(f"  - {t[0]}: {count} rows")
    con.close()
except Exception as e:
    print(f"Error opening database: {e}")
    print("Make sure the pipeline completed successfully.")